In [ ]:
# Handling sequential data using CNN wiht 1 D conv
# Data in this case sequential and model prediect a spefic sequance  
# labels encode (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) to 0-5
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten
import os

In [ ]:
SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]

def load_signals(data_dir, split):
    signals_data = []

    for signal in SIGNALS:
        filename = f"{signal}_{split}.txt"
        filepath = os.path.join(
            data_dir, split, "Inertial Signals", filename
        )
        signals_data.append(np.loadtxt(filepath))

    # shape: (samples, 128, 9)
    return np.stack(signals_data, axis=-1)


def load_labels(data_dir, split):
    path = os.path.join(data_dir, split, f"y_{split}.txt")
    return np.loadtxt(path).astype(int) - 1  # labels: 0–5


In [ ]:
DATASET_PATH = "D:/IMPORTANT/ML/DL/8-temporal_CNN/UCI HAR Dataset"

X_train = load_signals(DATASET_PATH, "train")
y_train = load_labels(DATASET_PATH, "train") 

X_test = load_signals(DATASET_PATH, "test")
y_test = load_labels(DATASET_PATH, "test")

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

print(X_train[0].shape)
print(y_train[0])

In [ ]:
mean = X_train.mean(axis=(0,1), keepdims=True)
std = X_train.std(axis=(0,1), keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std


In [ ]:
model = Sequential([
    Conv1D(64, 3, activation="relu", input_shape=(128, 9)),
    MaxPooling1D(2),

    Conv1D(128, 3, activation="relu"),
    #tf.keras.backend.batch_normalization(),
    MaxPooling1D(2),

    Conv1D(256, 3, activation="relu"),
    #tf.keras.backend.batch_normalization(),
    MaxPooling1D(2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(6, activation="softmax")
])

model.summary()


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=1
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")
